<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/30_num_int/20_second_order.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)



In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np
import numpy.linalg as nl
# 기호 연산 기능 추가
# Add symbolic operation capability
import sympy as sym



In [ ]:
sym.init_printing()



# 2차 적분<br>Second Order Numerical Integral



이 장에서는 $\int_0^1 e^x \, dx$를 주요 예제로 삼아 0차 적분 → 사다리꼴 → Simpson (이 노트북) 의 정확도 향상을 비교한다. 마지막에는 부정적분이 닫힌 형식으로 표현되지 않는 사례 (Bessel 함수 $I_0$, `50_exp_cos`) 도 다룬다.<br>
Throughout this chapter we use $\int_0^1 e^x \, dx$ as the running example to compare 0th-order → Trapezoidal → Simpson (this notebook). The chapter closes with an integral whose antiderivative has no closed form (Bessel $I_0$, in `50_exp_cos`).


부정적분이 알려져 있어 엄밀해를 비교 기준으로 쓸 수 있다.<br>
The antiderivative is known, so we have an exact reference value for comparison.

$$
\int_0^1 e^x \, dx = \left[ e^x \right]_0^1 = e - 1 \approx 1.7183
$$


In [ ]:
x_curve = np.linspace(0, 1, 100)
y_curve = np.exp(x_curve)

plt.fill_between(x_curve, y_curve, alpha=0.3)
plt.plot(x_curve, y_curve, label=r'$f(x) = e^x$')
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc=0)
plt.grid(True)


아래에서는 3 지점의 함수값을 이용하는 Simpson 규칙을 유도한다. 유도 과정의 시각적 표현은 반원을 쓰지만, 결과로 얻는 Simpson 공식은 매끄러운 적분 일반에 적용된다.<br>
Below we derive Simpson's rule, which uses function values at three points. The derivation uses a half circle as visual scaffolding, but the resulting Simpson formula applies to any smooth integrand.


In [ ]:
import plot_num_int as pi



In [ ]:
r = pi.radius_of_half_circle_area(1)



## 심슨 규칙<br>Simpson's Rule



마찬가지로 일정 간격으로 $x$ 좌표를 나누어 보자.<br>
Same as before, let's divide $x$ coordinates in a constant interval.



In [ ]:
n = 10

pi.plot_half_circle_with_stems(n, 1)



마지막 두 구간을 생각해 보기로 하자.<br>
Let's just think about the last two segments.



In [ ]:
n = 10

pi.plot_half_circle_with_stems(n, 1)
x_array, y_plus = pi.get_half_circle_xy_theta_space(1)
x_array_bar, y_array_bar = pi.get_half_circle_xy_linspace(n, 1)

# 마지막 두 구간에 해당하는 x y 값을 선택
# Choose x y values of the last two intervals
x_last_two_array = x_array[x_array_bar[-3] < x_array]
y_last_two_array = y_plus[x_array_bar[-3] < x_array]

plt.fill_between(x_last_two_array, y_last_two_array, color='orange')

plt.axis('equal')
plt.grid(True)



해당 넓이를 구하기 위해, 이 세 점을 지나는 2차 다항식을 찾아서 적분할 수 있을 것이다<br>
To get the area, we would be able to find a second order polynomial passing through these three points and integrate.


문제를 좀 더 쉽게 만들기 위해 해당 면적을 원점 주위로 평행 이동 시켜 보자.<br>
To make the problem simpler, let's translate the area around the origin.



In [ ]:
delta_x = x_array_bar[1]-x_array_bar[0]

plt.plot(x_array, y_plus, alpha=0.0)
plt.plot(x_array_bar[-3:], y_array_bar[-3:], '.')

# 마지막 두 구간을 표시
# Indicate last two intervals
plt.fill_between(x_last_two_array, y_last_two_array, color='orange')

# x 좌표 표시
# Indicate x coordinates
plt.text(x_last_two_array[0], -0.1, '$x_{n-2}$', horizontalalignment='center')
plt.text(x_last_two_array[-1], -0.1, '$x_{n}$', horizontalalignment='center')

# y 좌표 표시
# Indicate x coordinates
plt.text(x_array_bar[-3], y_array_bar[-3], '$f(x_{n-2})$', horizontalalignment='center', verticalalignment='bottom')
plt.text(x_array_bar[-2], y_array_bar[-2], '$f(x_{n-1})$', horizontalalignment='center', verticalalignment='bottom')
plt.text(x_array_bar[-1], y_array_bar[-1], '$f(x_{n})$', horizontalalignment='center', verticalalignment='bottom')


# 평행이동한 면적
# Translated Area
plt.plot(x_array_bar[-3:]-x_array_bar[-2], y_array_bar[-3:], '.')
plt.fill_between(x_last_two_array-x_array_bar[-2], y_last_two_array, color="purple")

# x 좌표 표시
# Indicate x coordinates
plt.text(-delta_x, -0.1, '$-\\Delta x$', horizontalalignment='center')
plt.text(delta_x, -0.1, '$+\\Delta x$', horizontalalignment='center')

# y 좌표 표시
# Indicate x coordinates
plt.text(-delta_x, y_array_bar[-3], '$y_0$', horizontalalignment='center', verticalalignment='bottom')
plt.text(          0, y_array_bar[-2], '$y_1$', horizontalalignment='center', verticalalignment='bottom')
plt.text(+delta_x, y_array_bar[-1], '$y_2$', horizontalalignment='center', verticalalignment='bottom')

plt.axis('equal')
plt.grid(True)



$$
y=a_0 x^2 + a_1 x + a_2
$$



원래 위치의 면적과 평행이동한 면적은 같다.<br>The translate area and the original area are equivalent.



평행이동한 면적의 세 점을 살펴 보자.<br>Let's take a look at the three points of the translated area.



$$
\begin{align}
    p_0&=\left(-\Delta x, y_0\right) \\
    p_1&=\left(0, y_1\right) \\
    p_2&=\left(\Delta x, y_2\right)
\end{align}
$$



In [ ]:
delta_x, y_m, y_0, y_p = sym.symbols('Delta_x, y_0, y_1, y_2', real=True)



In [ ]:
points = (-delta_x, y_m), (0, y_0), (delta_x, y_p)



In [ ]:
points



2차 다항식은 다음과 같은 형태를 가진다.<br>
A second order polynomial would take following form.



In [ ]:
a0, a1, a2, x = sym.symbols('a0, a1, a2, x', real=True)
f = a0 * x**2 + a1 * x + a2



In [ ]:
f



위 세 점을 모두 지나는 2차 곡선을 생각해 보자.<br>Let's think about a second order polynomial passing all three points above.


$$
\begin{align}
    y_0&=a_0 \left(-\Delta x\right)^2 + a_1 \left(-\Delta x\right) + a_2 \\
    y_1&=a_2 \\
    y_2&=a_0 \left(\Delta x\right)^2 + a_1 \left(\Delta x\right) + a_2
\end{align}
$$



In [ ]:
eq_points = [sym.Eq(p[-1], f.subs(x, p[0])) for p in points]



In [ ]:
eq_points



계수 $a_i$에 관하여 풀어 보자.<br>Let's try to solve for the coefficients $a_i$.



In [ ]:
a_sol = sym.solve(eq_points, (a0, a1, a2))



In [ ]:
a_sol



## 2차 다항식의 정적분<br>Definite Integral of a Second Order Polynomial



이제 $f(x)$를 $x$에 관하여 $-\Delta x$ 부터 $\Delta x$까지 적분해 보자.<br>Now let's integrate $f(x)$ about $x$ from $-\Delta x$ to $\Delta x$.



In [ ]:
integral = sym.integrate(f, (x, -delta_x, delta_x))



In [ ]:
integral



계수를 대입하고 정리해 보자.<br>Let's substitute the coefficients and simplfy.



In [ ]:
simpson = sym.simplify(integral.subs(a_sol))



In [ ]:
simpson



예를 들어 C 언어 코드로는 다음과 같이 가능하다<br>For example, in C programming language, following expression would be possible.



In [ ]:
sym.ccode(simpson)



## 심슨 규칙 구현<br>Implementing Simpson's Rule



한번에 두 구간의 면적을 계산한다.<br>
In one iteration, calculate the area of two intervals.



$$
    Area = F_0 + F_2 + \ldots + F_{n-2}
$$



$$
    F_k = \frac{\Delta x}{3}\left[f(x_k)+4 \cdot f(x_{k+1}) + f(x_{k+2})\right]
$$



In [ ]:
def get_delta_x(xi, xe, n):
    return (xe - xi)/n



In [ ]:
def num_int_2(f, xi, xe, n_partition, b_verbose=False):
    """
    f : function to integrate f(x)
    xi : start of integration
    xe : end of integration
    n_partition : number of partitions within the interval
    """
    # 구간의 갯수를 항상 짝수로 한다.
    # Always use even number of intervals

    if n_partition % 2:
        n_partition += 1

    delta_x = get_delta_x(xi, xe, n_partition)
    assert np.isclose((xi + delta_x*n_partition), xe), ((xi + delta_x*n_partition), xe)

    # delta_x 값이 너무 작은 경우
    # if delta_x is too small
    if 1e-7 > abs(delta_x):
        raise ValueError(f'delta_x(delta_x:g) too small')

    x_array = np.linspace(xi, xe, n_partition+1)

    assert np.isclose(abs(x_array[1] - x_array[0]), delta_x), (
        f"\ndelta_x = {delta_x} "
        f"\nx_array[1] - x_array[0] = {x_array[1] - x_array[0]}"
    )

    delta_x_third = delta_x / 3.0

    integration_result = 0.0
    xp = x_array[0]
    y0 = f(xp)

    for i in range(1, n_partition, 2):
        x1 = x_array[i]
        x2 = x_array[i+1]

        y1 = f(x1)
        y2 = f(x2)

        area_i = delta_x_third * (y0 + 4*y1 + y2)

        if b_verbose:
          print(f'x[{i-1}] = {xp}, x[{i}] = {x1}, x[{i+1}] = {x2}, area_i = {area_i}')

        xp, y0 = x2, y2
        integration_result += area_i

    return integration_result


In [ ]:
n = 10
result_exp = num_int_2(np.exp, 0.0, 1.0, n, b_verbose=True)
print('result =', result_exp)
print('exact  =', np.e - 1.0)


In [ ]:
n = 100
result_exp = num_int_2(np.exp, 0.0, 1.0, n)
print('result =', result_exp)
print('error  =', abs(result_exp - (np.e - 1.0)))


In [ ]:
%timeit -n 100 result_exp = num_int_2(np.exp, 0.0, 1.0, n)


### 수렴 차수 확인<br>Convergence order check

$f(x) = e^x$ 의 4차 도함수 $f^{(4)} = e^x$ 는 $[0, 1]$ 에서 유계이므로 절단 오차 $O(h^4)$ 가 그대로 나타나야 한다. 분할 수 $n$ 을 2 의 거듭제곱으로 늘려 가며 오차를 측정하고 log-log 그림에서 기울기를 확인해 보자.<br>
Since $f^{(4)} = e^x$ is bounded on $[0, 1]$, the truncation-error rate $O(h^4)$ should be observed empirically. Sweep $n$ as powers of two, measure the error, and check the slope on a log-log plot.


In [ ]:
n_values = 2 ** np.arange(1, 11)   # 2, 4, 8, ..., 1024
exact = np.e - 1.0
errors_exp = np.array([
    abs(num_int_2(np.exp, 0.0, 1.0, int(n)) - exact)
    for n in n_values
])

plt.loglog(n_values, errors_exp, 'o-', label=r'measured error on $e^x$')

# reference slope -4, anchored at n=8
anchor = 2
ref4 = errors_exp[anchor] * (n_values[anchor] / n_values) ** 4.0
plt.loglog(n_values, ref4, 'k--', alpha=0.5, label=r'reference $\propto n^{-4}$')

plt.xlabel('n (number of partitions)')
plt.ylabel('absolute error')
plt.title(r'Simpson 1/3 convergence on $e^x$ over $[0, 1]$')
plt.legend(loc=0)
plt.grid(True, which='both', alpha=0.5)
plt.show()

# slope estimate (drop noisy tail at float64 ULP)
mask = errors_exp > 1e-14
slope = np.polyfit(np.log(n_values[mask]), np.log(errors_exp[mask]), 1)[0]
print(f'measured slope = {slope:.2f}   (predicted -4 for smooth f)')


### 동적 탐색<br>Interactive Exploration

분할 수 $n$ (짝수) 을 바꾸어 가며 두 구간 묶음마다 적합되는 2차 다항식이 $e^x$ 곡선에 어떻게 근접하는지, 오차가 얼마나 빠르게 줄어드는지 직접 보자.<br>
Sweep an even number of partitions $n$ and watch the per-pair parabolic fits hug the $e^x$ curve &mdash; and how quickly the error shrinks.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_2_exp(n):
    if n % 2:
        n += 1   # Simpson 1/3 은 짝수 분할 / Simpson 1/3 needs even n

    f = np.exp
    xi_loc, xe_loc = 0.0, 1.0
    analytic = np.e - 1.0

    # 부드러운 곡선 / smooth curve
    x_curve = np.linspace(xi_loc, xe_loc, 256)
    y_curve = f(x_curve)
    plt.fill_between(x_curve, y_curve, alpha=0.3)

    # 분할점 / partition vertices
    x_bar = np.linspace(xi_loc, xe_loc, n + 1)
    y_bar = f(x_bar)

    # 두 구간 묶음마다의 포물선 적합 / parabolic fit per pair of intervals
    for k in range(0, n, 2):
        xs = x_bar[k:k+3]
        ys = y_bar[k:k+3]
        x_fit = np.linspace(xs[0], xs[-1], 32)
        coef = np.polyfit(xs, ys, 2)
        y_fit = np.polyval(coef, x_fit)
        plt.fill_between(x_fit, y_fit, alpha=0.5, edgecolor='k')

    plt.plot(x_bar, y_bar, 'ko', markersize=3)

    area = num_int_2(f, xi_loc, xe_loc, n)
    plt.title(f'n = {n},  Area = {area:.6f},  Error = {abs(area - analytic):.2e}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.show()


if _ci:
    plot_num_int_2_exp(10)
else:
    interact(plot_num_int_2_exp, n=IntSlider(min=2, max=200, step=2, value=10,
                                              description='n :'));


심슨 1/3 규칙의 절단 오차는 $-\frac{(b-a)}{180}\,h^4\,f^{(4)}(\xi)$ 이다. $n$ 을 두 배로 하면 오차는 약 $1/16$ 로 줄어든다 &mdash; 같은 연산량에서 사다리꼴(약 $1/4$)보다 훨씬 빠른 수렴을 보인다.<br>
Simpson's 1/3 rule has truncation error $-\frac{(b-a)}{180}\,h^4\,f^{(4)}(\xi)$. Doubling $n$ shrinks the error by roughly $1/16$ &mdash; much faster than the trapezoid rule's $1/4$ at the same cost.



## 특이 사례<br>Special cases

위 매끄러운 사례에서는 $f^{(4)}$ 가 유계여서 $O(h^4)$ 수렴 차수가 그대로 나타났다. 아래 사례들은 적분 구간 양 끝값이 같거나 (반원, $\cos$ 반 주기) 끝점에서 도함수가 발산해서 ($\sqrt{}$ 특이성), 사다리꼴 규칙과 Simpson 규칙의 차이가 흥미롭게 나타난다.<br>
The smooth case above showed the textbook $O(h^4)$ rate because $f^{(4)}$ is bounded. The cases below have either matched endpoint values (half circle, $\cos$ half period) or divergent derivatives at the endpoints (square-root singularity), leading to interesting deviations between the trapezoid and Simpson rules.


### 반원<br>Half circle

다시 면적 1인 반원을 생각해 보자.<br>
Let's revisit the half circle with area 1.


In [ ]:
pi.plot_a_half_circle_of_area(1)
pi.axis_equal_grid_True()



In [ ]:
n = 10
result_half = num_int_2(pi.half_circle, -r, r, n, b_verbose=True)
print('result =', result_half)


In [ ]:
n = 100
result_half = num_int_2(pi.half_circle, -r, r, n)
print('result =', result_half)
print('error  =', abs(result_half - 1.0))


In [ ]:
%timeit -n 100 result_half = num_int_2(pi.half_circle, -r, r, n)


$f(x) = \sqrt{r^2 - x^2}$ 는 끝점 ($x = \pm r$) 에서 도함수가 발산한다. $f^{(4)}$ 가 유계가 아니므로 위에서 본 $O(h^4)$ 보장이 그대로 성립하지 않고, 실제 측정 기울기는 $n^{-3/2}$ 에 가깝다.<br>
$f(x) = \sqrt{r^2 - x^2}$ has divergent derivatives at the endpoints ($x = \pm r$). Since $f^{(4)}$ is not bounded, the $O(h^4)$ guarantee above no longer holds; the measured slope is closer to $n^{-3/2}$.


In [ ]:
n_values = 2 ** np.arange(2, 11)   # 4, 8, ..., 1024
exact_half = 1.0   # unit area by construction
errors_half = np.array([
    abs(num_int_2(pi.half_circle, -r, r, int(n)) - exact_half)
    for n in n_values
])

plt.loglog(n_values, errors_half, 'o-', label='measured error on half circle')

# reference slopes for visual comparison
anchor = 0
ref4  = errors_half[anchor] * (n_values[anchor] / n_values) ** 4.0
ref32 = errors_half[anchor] * (n_values[anchor] / n_values) ** 1.5
plt.loglog(n_values, ref4,  'k--', alpha=0.4, label=r'$n^{-4}$ (textbook)')
plt.loglog(n_values, ref32, 'r--', alpha=0.4, label=r'$n^{-3/2}$ (observed)')

plt.xlabel('n (number of partitions)')
plt.ylabel('absolute error')
plt.title('Simpson 1/3 on half circle: endpoint singularity degrades rate')
plt.legend(loc=0)
plt.grid(True, which='both', alpha=0.5)
plt.show()

slope_half = np.polyfit(np.log(n_values), np.log(errors_half), 1)[0]
print(f'measured slope = {slope_half:.2f}   (textbook would predict -4)')


#### 동적 탐색<br>Interactive Exploration

반원에서 Simpson 규칙이 어떻게 동작하는지 분할 수 $n$ 을 바꾸어 가며 살펴 보자.<br>
Watch Simpson's rule on the half circle as $n$ changes.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_2(n):
    # 단위 면적 반원 / unit-area half-circle: r = sqrt(2/pi)
    r = np.sqrt(2.0 / np.pi)
    def half_circle(x):
        return np.sqrt(np.abs(r**2 - x**2))

    # 짝수로 강제 / coerce to even
    if n % 2:
        n += 1

    # 부드러운 곡선 / smooth integrand
    x_curve = np.linspace(-r, r, 256)
    y_curve = half_circle(x_curve)

    # 분할점 / partition vertices
    x_bar = np.linspace(-r, r, n + 1)
    y_bar = half_circle(x_bar)

    plt.fill_between(x_curve, y_curve, alpha=0.3)

    # 두 구간을 묶어 2차 다항식 적합 후 채우기
    # Fit a parabola over each pair of intervals and fill
    for k in range(0, n, 2):
        xs = x_bar[k:k+3]
        ys = y_bar[k:k+3]
        # numpy.polyfit returns highest power first / 최고차항부터 반환
        c2, c1, c0 = np.polyfit(xs, ys, 2)
        x_par = np.linspace(xs[0], xs[2], 32)
        y_par = c2*x_par**2 + c1*x_par + c0
        plt.fill_between(x_par, y_par, alpha=0.5, edgecolor='k')

    area = num_int_2(half_circle, -r, r, n)
    plt.title(f'n = {n},  Area = {area:.6f},  Error = {abs(area - 1):.2e}')
    plt.axis('equal')
    plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_num_int_2(10)
else:
    interact(
        plot_num_int_2,
        n=IntSlider(min=2, max=200, step=2, value=10, description='n :'),
    );


### $cos \theta$의 반 주기<br>Half period of $cos \theta$



In [ ]:
import numpy as np
def get_poly(x_list, y_list):
  # https://numpy.org/doc/stable/reference/routines.polynomials.html
  return np.polynomial.Polynomial.fit(x_list, y_list, deg=2)



In [ ]:
def calc_poly(x_list, y_list):
  x_array = np.linspace(x_list[0], x_list[2])
  return x_array, get_poly(x_list, y_list)(x_array)



In [ ]:
theta_deg = np.arange(180+1)
theta_rad = np.deg2rad(theta_deg)

s = np.sin(theta_rad)
c = np.cos(theta_rad)

# total area
# 전체 면적
plt.fill_between(theta_deg, c, color="C1", label="cos")

x_array_bar = np.linspace(0, 180, 10+1)
y_array_bar = np.cos(np.deg2rad(x_array_bar))

plt.xticks(x_array_bar)

x_gen = zip(
  x_array_bar[:-2:2],
  x_array_bar[1:-1:2],
  x_array_bar[2::2],
)
y_gen = zip(
  y_array_bar[:-2:2],
  y_array_bar[1:-1:2],
  y_array_bar[2::2],
)

# 2구간씩의 면적
for x_list, y_list in zip(x_gen, y_gen):
  plt.fill_between(*calc_poly(x_list, y_list), alpha=0.5)

plt.xlabel(r"$\theta(deg)$")
plt.grid(True)



In [ ]:
n = 10
result_cos = num_int_2(np.cos, 0, np.pi, n, b_verbose=True)
print('result =', result_cos)



In [ ]:
theta_deg = np.arange(180+1)
theta_rad = np.deg2rad(theta_deg)

s = np.sin(theta_rad)
c = np.cos(theta_rad)

# total area
# 전체 면적
plt.fill_between(theta_deg, c, color="C1", label="cos")

x_array_bar = np.linspace(0, 180, 100+1)
y_array_bar = np.cos(np.deg2rad(x_array_bar))

plt.vlines(x_array_bar, 0, y_array_bar)
plt.xticks(x_array_bar[::10])

x_gen = zip(
  x_array_bar[:-2:2],
  x_array_bar[1:-1:2],
  x_array_bar[2::2],
)
y_gen = zip(
  y_array_bar[:-2:2],
  y_array_bar[1:-1:2],
  y_array_bar[2::2],
)

# areas of two intervals
# 2구간씩의 면적
for x_list, y_list in zip(x_gen, y_gen):
  plt.fill_between(*calc_poly(x_list, y_list), alpha=0.5)

plt.xlabel(r"$\theta(deg)$")
plt.grid(True)



In [ ]:
n = 100
result_cos = num_int_2(np.cos, 0, np.pi, n)
print('result =', result_cos)



#### 동적 탐색<br>Interactive Exploration


분할 수 $n$ (짝수) 을 바꾸어 가며 두 구간 묶음마다의 2차 다항식이 cos 곡선에 어떻게 근접하는지 보자. 사다리꼴과 마찬가지로 $\int_0^\pi \cos\theta\,d\theta = 0$ 이 cos 의 반대칭에서 나오고, 심슨 규칙도 이 대칭성을 보존한다.<br>
Sweep an even $n$ and watch the per-pair parabolic fits hug the cosine curve. As with the trapezoid rule, $\int_0^\pi \cos\theta\,d\theta = 0$ from the antisymmetry of $\cos$, which Simpson's rule also respects.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_num_int_2_cos(n):
    if n % 2:
        n += 1   # Simpson 1/3 은 짝수 분할 / Simpson 1/3 needs even n

    # 부드러운 cos 곡선 / smooth cosine curve
    theta_deg_curve = np.linspace(0, 180, 256)
    y_curve = np.cos(np.deg2rad(theta_deg_curve))

    # 분할점 / partition vertices
    x_bar_deg = np.linspace(0, 180, n + 1)
    y_bar = np.cos(np.deg2rad(x_bar_deg))

    plt.fill_between(theta_deg_curve, y_curve, color="C1", alpha=0.3)

    # 두 구간씩 2차 다항식 적합 후 채우기
    # Fit a parabola over each pair of intervals and fill
    for k in range(0, n, 2):
        xs = x_bar_deg[k:k+3]
        ys = y_bar[k:k+3]
        c2, c1, c0 = np.polyfit(xs, ys, 2)
        x_par = np.linspace(xs[0], xs[2], 32)
        y_par = c2*x_par**2 + c1*x_par + c0
        plt.fill_between(x_par, y_par, alpha=0.5, edgecolor='k')

    area = num_int_2(np.cos, 0, np.pi, n)
    plt.title(f'n = {n},  Area = {area:+.6f}  (exact = 0)')
    plt.xlabel(r"$\theta(deg)$")
    plt.grid(True)
    plt.show()


if _ci:
    plot_num_int_2_cos(10)
else:
    interact(
        plot_num_int_2_cos,
        n=IntSlider(min=2, max=200, step=2, value=10, description='n :'),
    );


### 1/4 원<br>A quarter circle



In [ ]:
n = 10
result_quarter = num_int_2(pi.half_circle, -r, 0, n, b_verbose=True)
print('result =', result_quarter)



In [ ]:
n = 10
result_quarter = num_int_2(pi.half_circle, 0, r, n, b_verbose=True)
print('result =', result_quarter)



Again, let's compare with 0th order result.<br>
마찬가지로, 0차 적분 결과와 비교해 보자.



In [ ]:
n = 100
result_quarter = num_int_2(pi.half_circle, -r, 0, n)
print('result =', result_quarter)



## 연습 문제<br>Exercises



Try this 1: Using the 2nd order numerical integration, calculate the following. Please compare with the exact solution and the 0th & 1st order results<br>도전 과제 1: 2차 적분을 이용하여 다음을 계산하시오. 이론값, 0, 1차 적분 값과 비교하시오

$$
\int_0^{2\pi}cos \theta d\theta
$$



Try this 2 : Using the example of half circle with area 1, compare errors with zeroth and first order integrations.<br>도전 과제 2 : 넓이 1인 반원의 예로 0차, 1차 적분과의 오차를 비교하시오.



Try this 3 : Calculate the half of area of an ellipse with long diameter 4 and short diameter 2 using the Simpson's rule. [[wikipedia](https://en.wikipedia.org/wiki/Ellipse)]<br>도전 과제 3 : 긴 지름 4, 짧은 지름 2인 타원의 면적의 절반을 심슨법으로 계산하시오. [[위키피디아](https://ko.wikipedia.org/wiki/%ED%83%80%EC%9B%90)]



$$
    \frac{x^2}{4^2} + \frac{y^2}{2^2} = 1
$$



Try this 4: Using the 2nd order numerical integration, calculate the following.<br>도전 과제 4: 2차 적분을 이용하여 다음을 계산하시오.

$$
\left(2\int_0^{1}e^{-x^2} dx\right)^2
$$



Try this 5: Using the 2nd order numerical integration, calculate the following. Please compare with the result above.<br>도전 과제 4: 2차 적분을 이용하여 다음을 계산하시오. 위 결과와 비교하시오.

$$
\left(2\int_0^{10}e^{-x^2} dx\right)^2
$$



## 함수형 프로그래밍<br>Functional programming



$n$ 개의 간격에 대해 심슨 규칙 적용을 생각해 보자.<br>
Let's think about applying Simpson's rule over $n$ intervals.



$$
    Area = F_0 + F_2 + \ldots + F_{n-2}
$$



$$
    F_k = \frac{\Delta x}{3}\left[f(x_k)+4 \cdot f(x_{k+1}) + f(x_{k+2})\right]
$$



$$
\begin{align}
    Area &= \frac{\Delta x}{3}\left[f(x_0)+4 \cdot f(x_{1}) + f(x_{2})\right] \\
        &+ \frac{\Delta x}{3}\left[f(x_2)+4 \cdot f(x_{3}) + f(x_{4})\right] \\
        &+ \frac{\Delta x}{3}\left[f(x_4)+4 \cdot f(x_{5}) + f(x_{6})\right] \\
        & \ldots \\
        &+ \frac{\Delta x}{3}\left[f(x_{n-4})+4 \cdot f(x_{n-3}) + f(x_{n-2})\right] \\
        &+ \frac{\Delta x}{3}\left[f(x_{n-2})+4 \cdot f(x_{n-1}) + f(x_{n})\right] \\
\end{align}
$$



$$
\begin{align}
    Area &= \frac{\Delta x}{3}\left[f(x_0)+f(x_{n})\right] \\
        &+ \frac{4}{3}\Delta x \left[f(x_{1}) + f(x_{3}) + f(x_{5}) + \ldots + f(x_{n-3}) + f(x_{n-1})\right] \\
        &+ \frac{2}{3}\Delta x \left[f(x_{2}) + f(x_{4}) + f(x_{6}) + \ldots + f(x_{n-4}) + f(x_{n-2})\right] \\
\end{align}
$$



In [ ]:
def even_sum_func(f, xi, xe, delta_x):
    return sum(
        map(
            f,
            np.arange(xi+delta_x, xe-delta_x*0.5, delta_x*2),
        )
    )



In [ ]:
def odd_sum_func(f, xi, xe, delta_x):
    return sum(
        map(
            f,
            np.arange(xi+(delta_x*2) , xe-delta_x*0.5, delta_x*2),
        )
    )



In [ ]:
def num_int_2_functional(f, xi, xe, n):
    return (
        (get_delta_x(xi, xe, n) * (1.0/3)) * (
            f(xi) + f(xe)
            + 4 * even_sum_func(f, xi, xe, get_delta_x(xi, xe, n))
            + 2 * odd_sum_func(f, xi, xe, get_delta_x(xi, xe, n))
        )
    )



In [ ]:
n = 100
result_func = num_int_2_functional(np.exp, 0.0, 1.0, n)
print('result_func =', result_func)
print('error       =', abs(result_func - (np.e - 1.0)))


In [ ]:
assert 1e-7 > abs(num_int_2(np.exp, 0.0, 1.0, n) - result_func), \
    f'functional and procedural results should match'


In [ ]:
%timeit -n 100 result_func = num_int_2_functional(np.exp, 0.0, 1.0, n)


특이 사례 회귀 검사 / Smoke regression on the special case (half circle):


In [ ]:
# half_circle, r 은 위 '### 반원' 절에서 정의됨 / defined in '### 반원' above
result_func_smoke = num_int_2_functional(pi.half_circle, -r, r, 100)
assert 1e-3 > abs(result_func_smoke - 1.0), \
    f'half-circle smoke regression: result_func_smoke = {result_func_smoke}'


## 시험<br>Test



아래는 함수가 맞게 작동하는지 확인함<br>
Following cells verify whether the functions work correctly.



In [ ]:
# 주 예제 / primary example: e^x on [0, 1]
n = 100
exact = np.e - 1.0
assert 1e-6 > abs(num_int_2(np.exp, 0.0, 1.0, n) - exact), \
    f'num_int_2 on e^x: {num_int_2(np.exp, 0.0, 1.0, n)} vs {exact}'
assert 1e-6 > abs(num_int_2_functional(np.exp, 0.0, 1.0, n) - exact), \
    f'num_int_2_functional on e^x: {num_int_2_functional(np.exp, 0.0, 1.0, n)} vs {exact}'


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
r = np.sqrt(1.0 / np.pi)
n = 10
delta_x = r/n


def half_circle(x):
    return np.sqrt(r**2 - x ** 2)


assert 0.25 > num_int_2(half_circle, -r, 0, n)
assert 0.25 > num_int_2(half_circle, 0, r, n)
assert 0.25 > num_int_2_functional(half_circle, -r, 0, n)
assert 0.25 > num_int_2_functional(half_circle, 0, r, n)



In [ ]:
import math

epsilon = 0.005

assert math.isclose(4.0 * num_int_2(half_circle, -r, 0, n),            1.0, abs_tol=epsilon), (4.0 * num_int_2(half_circle, -r, 0, n))
assert math.isclose(4.0 * num_int_2(half_circle, 0, r, n),             1.0, abs_tol=epsilon), (4.0 * num_int_2(half_circle, 0, r, n))
assert math.isclose(4.0 * num_int_2_functional(half_circle, -r, 0, n), 1.0, abs_tol=epsilon), (4.0 * num_int_2_functional(half_circle, -r, 0, n))
assert math.isclose(4.0 * num_int_2_functional(half_circle, 0, r, n),  1.0, abs_tol=epsilon), (4.0 * num_int_2_functional(half_circle, 0, r, n))



## Final Bell<br>마지막 종



In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");

